# SFTTrainer Fine-Tuning on `entry2exit_df`

This notebook uses `trl.SFTTrainer` to train an instruction-style causal LM on:
- input: `entry2exit_df["input_text"]`
- output: `entry2exit_df["target_text"]`

Expected CSV:
- `./entry2exit.csv` with columns `input_text`, `target_text`
- optional `entry_date` for time-aware split

In [1]:
# If needed once:
# !pip install -q transformers datasets trl accelerate bitsandbytes sentencepiece peft

In [2]:
import inspect
import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
)
from trl import SFTTrainer
try:
    from trl import SFTConfig
except Exception:
    SFTConfig = None

print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training


Skipping import of cpp extensions due to incompatible torch version 2.9.1 for torchao version 0.16.0             Please see https://github.com/pytorch/ao/issues/2919 for more info
W0219 20:13:13.357000 51671 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Torch: 2.9.1
CUDA available: False


In [3]:
# 1) Load + clean dataframe from CSV
from pathlib import Path

ENTRY2EXIT_CSV_PATH = Path("./entry2exit.csv")
if not ENTRY2EXIT_CSV_PATH.exists():
    raise RuntimeError(
        f"CSV not found at {ENTRY2EXIT_CSV_PATH.resolve()}. "
        "Generate it from your other notebook with: "
        "entry2exit_df.to_csv('entry2exit.csv', index=False)."
    )

entry2exit_df = pd.read_csv(ENTRY2EXIT_CSV_PATH)
required_cols = {"input_text", "target_text"}
if not required_cols.issubset(entry2exit_df.columns):
    raise RuntimeError(f"CSV must contain columns {required_cols}. Found: {list(entry2exit_df.columns)}")

sft_df = entry2exit_df.copy()
sft_df = sft_df.dropna(subset=["input_text", "target_text"]).reset_index(drop=True)
sft_df = sft_df[
    sft_df["input_text"].astype(str).str.len().gt(0)
    & sft_df["target_text"].astype(str).str.len().gt(0)
].reset_index(drop=True)

if "entry_date" in sft_df.columns:
    sft_df["entry_date"] = pd.to_datetime(sft_df["entry_date"], errors="coerce")
    sft_df = sft_df.sort_values("entry_date").reset_index(drop=True)

split_idx = int(len(sft_df) * 0.90)
train_df = sft_df.iloc[:split_idx].copy()
val_df = sft_df.iloc[split_idx:].copy()

print('Loaded CSV:', ENTRY2EXIT_CSV_PATH.resolve())
print('Total rows:', len(sft_df))
print('Train rows:', len(train_df))
print('Val rows:', len(val_df))
display(train_df[["input_text", "target_text"]].head(2))


Loaded CSV: /Users/johnnylee/PycharmProjects/optimal_trader/entry2exit.csv
Total rows: 2197
Train rows: 1977
Val rows: 220


,input_text,target_text
0,Symbol=AA EntryDate=1997-05-22 Open=27.11 High...,Symbol=AA ExitDate=1999-01-04 Open=28.67 High=...
1,Symbol=AA EntryDate=1997-06-13 Open=29.78 High...,Symbol=AA ExitDate=1997-06-26 Open=27.9 High=2...


In [4]:
# 2) Build instruction format text for SFT
PROMPT_TEMPLATE = """### Task:
Given entry market state, generate the exit state text.

### Input:
{input_text}

### Output:
{target_text}"""


def format_example(row: pd.Series) -> str:
    return PROMPT_TEMPLATE.format(
        input_text=str(row["input_text"]),
        target_text=str(row["target_text"]),
    )

train_sft = train_df.copy()
val_sft = val_df.copy()
train_sft["text"] = train_sft.apply(format_example, axis=1)
val_sft["text"] = val_sft.apply(format_example, axis=1)

train_ds = Dataset.from_pandas(train_sft[["text"]], preserve_index=False)
val_ds = Dataset.from_pandas(val_sft[["text"]], preserve_index=False)

print(train_ds[0]["text"][:600])

### Task:
Given entry market state, generate the exit state text.

### Input:
Symbol=AA EntryDate=1997-05-22 Open=27.11 High=27.39 Low=26.82 Close=27.06 Volume=539396 Ret1d=0 Ret2d=-0.01 Ret3d=-0.02 Ret5d=-0.02 Ret10d=0.01 Ret20d=0.04 Ret63d=0.02 Ret126d=0.14 Ret252d=0.17 CumRet5d=-0.02 CumRet10d=0.01 CumRet20d=0.04 CumRet63d=0.02 DistSMA5=-0.01 SMASlope5=-0.1 DistSMA10=-0.01 SMASlope10=0.02 DistSMA20=0.01 SMASlope20=0.05 DistSMA50=0.03 SMASlope50=-0.02 DistSMA100=0.03 SMASlope100=0.03 DistSMA200=0.11 SMASlope200=0.02 DistEMA12=0 DistEMA26=0.01 DistEMA50=0.02 MACD=0.37 MACDSignal=0.36 MACDHist


In [5]:
# 3) Load model + tokenizer + PEFT (LoRA)
# Choose a model that fits your hardware. Examples:
# model_name = "Qwen/Qwen2.5-0.5B-Instruct"
# model_name = "Qwen/Qwen2.5-1.5B-Instruct"
model_name = "Qwen/Qwen2.5-0.5B-Instruct"

use_4bit = torch.cuda.is_available()

if use_4bit:
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
        device_map="auto",
        load_in_4bit=True,
    )
    model = prepare_model_for_kbit_training(model)
else:
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float32,
    )
    model = model.to("cpu")

tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# LoRA target modules for Qwen-style attention/MLP projections
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


`torch_dtype` is deprecated! Use `dtype` instead!


trainable params: 8,798,208 || all params: 502,830,976 || trainable%: 1.7497


In [ ]:
# 4) Train with SFTTrainer
max_seq_length = 512

base_train_kwargs = dict(
    output_dir="./artifacts/sfttrainer_entry2exit",
    num_train_epochs=10,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    per_device_eval_batch_size=1,
    learning_rate=2e-5,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=200,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=2,
    report_to=[],
    fp16=torch.cuda.is_available() and not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
)

if hasattr(model, "config") and hasattr(model.config, "use_cache"):
    model.config.use_cache = False

if SFTConfig is not None:
    sft_sig = inspect.signature(SFTConfig.__init__).parameters
    sft_kwargs = dict(base_train_kwargs)

    if "dataset_text_field" in sft_sig:
        sft_kwargs["dataset_text_field"] = "text"
    if "packing" in sft_sig:
        sft_kwargs["packing"] = False
    if "max_seq_length" in sft_sig:
        sft_kwargs["max_seq_length"] = max_seq_length
    elif "max_length" in sft_sig:
        sft_kwargs["max_length"] = max_seq_length

    training_args = SFTConfig(**sft_kwargs)
else:
    training_args = TrainingArguments(**base_train_kwargs)

sig = inspect.signature(SFTTrainer.__init__).parameters
trainer_kwargs = {
    "model": model,
    "args": training_args,
    "train_dataset": train_ds,
    "eval_dataset": val_ds,
}

if "processing_class" in sig:
    trainer_kwargs["processing_class"] = tokenizer
elif "tokenizer" in sig:
    trainer_kwargs["tokenizer"] = tokenizer

if "dataset_text_field" in sig:
    trainer_kwargs["dataset_text_field"] = "text"
if "max_seq_length" in sig:
    trainer_kwargs["max_seq_length"] = max_seq_length
elif "max_length" in sig:
    trainer_kwargs["max_length"] = max_seq_length
if "packing" in sig:
    trainer_kwargs["packing"] = False

trainer = SFTTrainer(**trainer_kwargs)

train_result = trainer.train()
print(train_result)


Adding EOS to train dataset:   0%|          | 0/1977 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1977 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/1977 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/220 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/220 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/220 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.
/Users/johnnylee/miniconda3/envs/optimal_trader/lib/python3.13/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
200,0.318300,0.321309,0.318871,409600.000000,0.879532
400,0.291800,0.292679,0.287872,819200.000000,0.887680


/Users/johnnylee/miniconda3/envs/optimal_trader/lib/python3.13/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


In [ ]:
# 5) Save
save_dir = "./artifacts/sfttrainer_entry2exit"
trainer.model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)
print('Saved model/tokenizer to:', save_dir)

In [ ]:
# 6) Inference helper + sample predictions
model.eval()

def generate_exit_text_sft(input_text: str, max_new_tokens: int = 160) -> str:
    prompt = f"""### Task:
Given entry market state, generate the exit state text.

### Input:
{input_text}

### Output:
"""
    inputs = tokenizer(prompt, return_tensors="pt")
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.inference_mode():
        out_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            num_beams=1,
            repetition_penalty=1.05,
            no_repeat_ngram_size=3,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    text = tokenizer.decode(out_ids[0], skip_special_tokens=True)
    if "### Output:" in text:
        text = text.split("### Output:", 1)[1].strip()
    return text

print("\n--- Sample predictions ---")
for i in range(min(3, len(val_df))):
    inp = str(val_df.iloc[i]["input_text"])
    tgt = str(val_df.iloc[i]["target_text"])
    pred = generate_exit_text_sft(inp)

    print(f"\n[{i}] INPUT:\n{inp[:350]}...")
    print(f"TARGET:\n{tgt[:350]}...")
    print(f"PRED:\n{pred[:350]}...")
